# Guidance strength experiments

Change **`GUIDANCE_SCALES`** and **`SEEDS`** below, then run the cells in order.

This notebook reuses the existing trained ClavaDDPM models. It does **not** train models or change the original configs, notebooks, or results.
Each trial generates all eight tables and saves its settings, samples, and scores under `results/experiments/`.

Start with one seed to explore. Repeat promising values with more seeds before choosing a winner.

In [1]:
import json
import pickle
import sys
from copy import deepcopy
from logging import INFO
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display as show
from midst_toolkit.common.config import ClavaDDPMMatchingConfig, ClavaDDPMSamplingConfig, GeneralConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER
from midst_toolkit.common.random import set_all_random_seeds
from midst_toolkit.evaluation.quality import CorrelationMatrixDifference
from midst_toolkit.models.clavaddpm.synthesizer import (
    clava_synthesizing,
    clava_synthesizing_matching_process,
)
from omegaconf import OmegaConf
from scipy.stats import ks_2samp


# Works from the repository root or this notebook's directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
MULTI_TABLE = ROOT / "implementations/tabular_data/multi_table"
from implementations.tabular_data.evaluation.preprocessing import syntheval_preprocess


TOOLKIT_LOGGER.setLevel(INFO)

/home/coder/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## 1. Choose the trials

Only guidance changes between trials. Keep batch size, sample scale, matching settings, models, and seed fixed for a fair first comparison.
`1.0` reproduces the current guidance setting; `0.0` disables its gradient contribution.

Completed trials are reused only when their saved settings and checkpoint file signatures match. Use a new `SWEEP_NAME` to start fresh.

In [2]:
GUIDANCE_SCALES = [0.0, 0.5, 1.0, 2.0, 4.0]
SEEDS = [42]  # Later: [42, 123, 456]
SWEEP_NAME = "guidance_strength_first_pass"

BASE_RESULTS = MULTI_TABLE / "results"
DATA_DIR = MULTI_TABLE / "data/berka"
SWEEP_DIR = BASE_RESULTS / "experiments" / SWEEP_NAME
cfg = OmegaConf.load(MULTI_TABLE / "synthesizing/config.yaml")

assert GUIDANCE_SCALES and all(np.isfinite(s) and s >= 0 for s in GUIDANCE_SCALES)
assert SEEDS and all(isinstance(s, int) and s >= 0 for s in SEEDS)
assert cfg.matching_config.unique_matching and not cfg.matching_config.no_matching, (
    "Use deterministic matching for this guidance sweep."
)
print(f"{len(GUIDANCE_SCALES) * len(SEEDS)} trials; outputs: {SWEEP_DIR}")
print(OmegaConf.to_yaml(cfg))
print("CUDA available:", torch.cuda.is_available())

5 trials; outputs: /home/coder/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results/experiments/guidance_strength_first_pass
sampling_config:
  sample_scale: 1.0
  batch_size: 200
  classifier_scale: 1.0
matching_config:
  num_matching_clusters: 1
  matching_batch_size: 1000
  unique_matching: true
  no_matching: false
random_seed: 42

CUDA available: True


## 2. Load the existing checkpoints

The checkpoint's original tables are the real-data reference, so evaluation uses the same preprocessed data that training saw.
Checkpoint files are opened for reading only. Models run in evaluation mode to disable classifier dropout.

In [3]:
with (DATA_DIR / "dataset_meta.json").open() as f:
    relation_order = [tuple(edge) for edge in json.load(f)["relation_order"]]

checkpoint_paths = [BASE_RESULTS / "cluster_ckpt.pkl"] + [
    BASE_RESULTS / "models" / f"{parent}_{child}_ckpt.pkl" for parent, child in relation_order
]
for path in checkpoint_paths:
    if not path.is_file():
        raise FileNotFoundError(f"Missing trained checkpoint: {path}")

# Detect changed source checkpoints when reusing completed trials.
checkpoint_signatures = {
    str(path.relative_to(MULTI_TABLE)): {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
    for path in checkpoint_paths
}
with checkpoint_paths[0].open("rb") as f:
    clustering = pickle.load(f)
tables = clustering["tables"]
group_lengths = clustering["all_group_lengths_prob_dicts"]
real_tables = {name: table.data[table.original_column_names].copy() for name, table in tables.items()}
domains = {
    name: {column: spec for column, spec in table.domain.items() if column in table.original_column_names}
    for name, table in tables.items()
}
models = {}
for edge, path in zip(relation_order, checkpoint_paths[1:], strict=True):
    with path.open("rb") as f:
        model = pickle.load(f)
    model.diffusion.eval()
    if model.classifier is not None:
        model.classifier.eval()
    models[edge] = model
print(f"Loaded {len(models)} models and {len(real_tables)} real tables. No training needed.")

Loaded 9 models and 8 real tables. No training needed.


## 3. Reuse the evaluation approach

For **every edge**, measure foreign-key validity, child-count similarity (including zero-child parents), and cross-table mixed correlation error using the same toolkit metric as the original evaluation notebook.
Numerical columns use KS distance; encoded categorical columns use total variation distance. Both distances range from 0 to 1; lower is better.

Cross-table RMSE normalizes correlation error by the number of feature pairs. Undefined correlations remain missing and are counted, rather than silently scored as perfect.
These are fidelity comparisons against training data, not held-out utility or privacy measurements.

In [4]:
def cross_table_error(parent_name, child_name, synthetic):
    """Reuse mixed correlations from multi_table_metrics.ipynb on one edge."""
    key = f"{parent_name}_id"
    feature_names, numerical, categorical = {}, [], []
    for name in (parent_name, child_name):
        feature_names[name] = [f"{name}__{column}" for column in domains[name]]
        for column, spec in domains[name].items():
            columns = numerical if spec["type"] == "continuous" else categorical
            columns.append(f"{name}__{column}")

    def joined(source):
        frames = []
        for name in (parent_name, child_name):
            features = list(domains[name])
            frames.append(source[name][[key, *features]].rename(columns={c: f"{name}__{c}" for c in features}))
        result = frames[1].merge(frames[0], on=key, how="inner", validate="many_to_one")
        result = result[numerical + categorical].copy()
        for column in categorical:
            # Normalize numeric category codes: 0 and 0.0 are the same category.
            result[column] = pd.to_numeric(result[column]).astype(float).astype(str)
        return result

    real_joined, synth_joined = joined(real_tables), joined(synthetic)
    n_pairs = len(feature_names[parent_name]) * len(feature_names[child_name])
    if len(synth_joined) < 2:
        return {"cross_table_rmse": np.nan, "undefined_pairs": n_pairs}
    real_encoded, synth_encoded = syntheval_preprocess(numerical, categorical, real_joined, synth_joined)
    metric = CorrelationMatrixDifference(
        categorical_columns=categorical,
        numerical_columns=numerical,
        compute_mixed_correlations=True,
        do_preprocess=False,
    )
    metric.compute(real_encoded, synth_encoded)
    matrices = metric.syntheval_metric.evaluate(mixed_corr=True, return_mats=True)
    difference = (
        (matrices["real_cor_mat"] - matrices["synt_cor_mat"])
        .loc[
            feature_names[parent_name],
            feature_names[child_name],
        ]
        .to_numpy(dtype=float)
    )
    return {
        "cross_table_rmse": float(np.sqrt(np.mean(difference**2))),
        "undefined_pairs": int((~np.isfinite(difference)).sum()),
    }


def evaluate(synthetic):
    """Score final keyed tables; return a compact summary and inspectable details."""
    column_scores, table_scores, edge_scores = [], [], []
    for name, real in real_tables.items():
        synth = synthetic[name]
        key = f"{name}_id"
        table_scores.append(
            {
                "table": name,
                "real_rows": len(real),
                "synthetic_rows": len(synth),
                "pk_unique_rate": float(synth[key].nunique() / len(synth)) if len(synth) else np.nan,
            },
        )
        for column, spec in domains[name].items():
            real_values = pd.to_numeric(real[column])
            synth_values = pd.to_numeric(synth[column])
            if spec["type"] == "continuous":
                distance = ks_2samp(real_values, synth_values).statistic if len(synth) else np.nan
                kind = "numerical_ks"
            else:
                real_freq, synth_freq = real_values.value_counts(normalize=True).align(
                    synth_values.value_counts(normalize=True),
                    fill_value=0,
                )
                distance = 0.5 * (real_freq - synth_freq).abs().sum() if len(synth) else np.nan
                kind = "categorical_tvd"
            column_scores.append({"table": name, "column": column, "metric": kind, "distance": float(distance)})

    for parent, child in relation_order:
        if parent is None:
            continue
        key = f"{parent}_id"
        real_parent, real_child = real_tables[parent], real_tables[child]
        synth_parent, synth_child = synthetic[parent], synthetic[child]
        real_counts = real_parent[key].map(real_child.groupby(key).size()).fillna(0)
        synth_counts = synth_parent[key].map(synth_child.groupby(key).size()).fillna(0)
        edge_scores.append(
            {
                "edge": f"{parent}->{child}",
                "fk_valid_rate": float(synth_child[key].isin(synth_parent[key]).mean()),
                "cardinality_ks": float(ks_2samp(real_counts, synth_counts).statistic) if len(synth_parent) else np.nan,
                **cross_table_error(parent, child, synthetic),
            },
        )

    edges, columns, sizes = pd.DataFrame(edge_scores), pd.DataFrame(column_scores), pd.DataFrame(table_scores)
    key = "account_id"
    real_join = real_tables["loan"].merge(real_tables["account"], on=key, validate="many_to_one")
    synth_join = synthetic["loan"].merge(synthetic["account"], on=key, validate="many_to_one")
    dates = ["account_date", "loan_date"]
    real_date_corr = real_join[dates].astype(float).corr().iloc[0, 1]
    synth_date_corr = synth_join[dates].astype(float).corr().iloc[0, 1]
    summary = {
        "min_fk_valid_rate": edges.fk_valid_rate.min(skipna=False),
        "min_pk_unique_rate": sizes.pk_unique_rate.min(skipna=False),
        "mean_cardinality_ks": edges.cardinality_ks.mean(skipna=False),
        "mean_cross_table_rmse": edges.cross_table_rmse.mean(skipna=False),
        "undefined_pairs": int(edges.undefined_pairs.sum()),
        "mean_numerical_ks": columns.loc[columns.metric == "numerical_ks", "distance"].mean(skipna=False),
        "mean_categorical_tvd": columns.loc[columns.metric == "categorical_tvd", "distance"].mean(skipna=False),
        "real_date_corr": real_date_corr,
        "synthetic_date_corr": synth_date_corr,
        "date_corr_error": abs(real_date_corr - synth_date_corr),
        "loan_rows": len(synthetic["loan"]),
    }
    # Pandas converts missing scores to JSON null, keeping metrics.json valid JSON.
    return {
        "summary": json.loads(pd.DataFrame([summary]).to_json(orient="records"))[0],
        "edges": json.loads(edges.to_json(orient="records")),
        "columns": json.loads(columns.to_json(orient="records")),
        "tables": json.loads(sizes.to_json(orient="records")),
    }

## 4. Run the guidance sweep

Each trial uses the full dataset size (`sample_scale` from the original synthesis config).
The toolkit saves pre-matching samples and feature-only CSVs. Its public matching function is repeated on those saved samples to preserve a final table dictionary **with keys** for evaluation.
The current matching settings are deterministic, so the repeated pass uses identical inputs and produces the same matches. No diffusion samples are regenerated.

Full generation can take several minutes per trial on a GPU. The first trial prints its duration to estimate the remaining work.
Completed trials are skipped on rerun. If a trial was interrupted, choose a new `SWEEP_NAME` before restarting.

In [ ]:
results = []
for seed in SEEDS:
    for scale in GUIDANCE_SCALES:
        run_dir = SWEEP_DIR / f"guidance_{scale:g}_seed{seed}"
        resolved = OmegaConf.to_container(cfg, resolve=True)
        resolved["sampling_config"]["classifier_scale"] = float(scale)
        resolved["random_seed"] = seed
        resolved["checkpoint_signatures"] = checkpoint_signatures
        resolved["model_mode"] = "eval"
        resolved["torch_version"] = str(torch.__version__)
        metrics_path = run_dir / "metrics.json"
        if metrics_path.exists():
            saved_config = OmegaConf.to_container(OmegaConf.load(run_dir / "config.yaml"), resolve=True)
            if saved_config != resolved:
                raise ValueError(f"Settings changed for {run_dir.name}. Choose a new SWEEP_NAME.")
            metrics = json.loads(metrics_path.read_text())
            print(f"Reusing completed trial: {run_dir.name}")
        else:
            # Never overwrite an existing or partially completed experiment.
            run_dir.mkdir(parents=True, exist_ok=False)
            OmegaConf.save(OmegaConf.create(resolved), run_dir / "config.yaml")
            set_all_random_seeds(seed)
            started = perf_counter()
            print(f"Starting {run_dir.name}", flush=True)
            _, synthesis_seconds, matching_seconds = clava_synthesizing(
                deepcopy(tables),
                relation_order,
                run_dir,
                models,
                GeneralConfig(
                    data_dir=DATA_DIR,
                    test_data_dir=DATA_DIR,
                    workspace_dir=run_dir,
                    exp_name="tables",
                    sample_prefix="synthetic",
                ),
                ClavaDDPMSamplingConfig(
                    batch_size=cfg.sampling_config.batch_size,
                    classifier_scale=float(scale),
                ),
                ClavaDDPMMatchingConfig(**cfg.matching_config),
                group_lengths,
                sample_scale=cfg.sampling_config.sample_scale,
            )
            with (run_dir / "before_matching/synthetic_tables.pkl").open("rb") as f:
                before_matching = pickle.load(f)
            # Repeat deterministic matching to retain keys dropped by CSV export.
            final_tables = clava_synthesizing_matching_process(
                before_matching,
                tables,
                relation_order,
                ClavaDDPMMatchingConfig(**cfg.matching_config),
            )
            with (run_dir / "synthetic_final_with_keys.pkl").open("wb") as f:
                pickle.dump(final_tables, f)
            metrics = evaluate(final_tables)
            metrics["summary"].update(
                {
                    "synthesis_seconds": synthesis_seconds,
                    "matching_seconds": matching_seconds,
                    "total_seconds": perf_counter() - started,
                },
            )
            metrics_path.write_text(json.dumps(metrics, indent=2, allow_nan=False) + "\n")
            print(f"Finished {run_dir.name}: {metrics['summary']['total_seconds'] / 60:.1f} minutes", flush=True)
        results.append({"guidance": float(scale), "seed": seed, **metrics["summary"]})
        show(
            pd.DataFrame(results)[
                [
                    "guidance",
                    "seed",
                    "mean_cross_table_rmse",
                    "date_corr_error",
                    "mean_numerical_ks",
                    "mean_categorical_tvd",
                    "min_fk_valid_rate",
                ]
            ],
        )

## 5. Compare the results

Lower is better for all error/distance columns; FK validity and PK uniqueness should both be **1.0**.
Compare cross-table error alongside column fidelity; stronger guidance can improve one and worsen the other.
`undefined_pairs` should be zero. A missing score is not a good score.

The existing saved baseline may differ because this notebook explicitly sets all models to evaluation mode.
Use the fresh `guidance=1` trial as the controlled baseline. One seed is exploratory; add seeds before drawing a firm conclusion.

In [ ]:
# Read saved scores: this cell can be rerun without regenerating data.
results = []
for metrics_path in sorted(SWEEP_DIR.glob("guidance_*/metrics.json")):
    settings = OmegaConf.load(metrics_path.parent / "config.yaml")
    metrics = json.loads(metrics_path.read_text())
    results.append(
        {
            "guidance": settings.sampling_config.classifier_scale,
            "seed": settings.random_seed,
            **metrics["summary"],
        },
    )
comparison = pd.DataFrame(results).sort_values(["guidance", "seed"])
show(comparison.round(4))
comparison.to_csv(SWEEP_DIR / "comparison.csv", index=False)

# Mean/std become useful when SEEDS contains more than one seed.
show(
    comparison.groupby("guidance")[
        [
            "mean_cross_table_rmse",
            "date_corr_error",
            "mean_numerical_ks",
            "mean_categorical_tvd",
        ]
    ]
    .agg(["mean", "std"])
    .round(4),
)

In [ ]:
# Inspect one trial's edge, column, or table scores without generating anything.
INSPECT_SCALE = 1.0
INSPECT_SEED = SEEDS[0]
inspect_dir = SWEEP_DIR / f"guidance_{INSPECT_SCALE:g}_seed{INSPECT_SEED}"
details = json.loads((inspect_dir / "metrics.json").read_text())
show(pd.DataFrame(details["edges"]))
show(pd.DataFrame(details["columns"]).sort_values("distance", ascending=False).head(10))
show(pd.DataFrame(details["tables"]))